# 🎬 CopyAir / NeuralShot - Google Colab Training & Inference Studio

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Serces19/copyair/blob/main/notebooks/copyair_colab_training.ipynb)

Bienvenido al entorno de ejecución en la nube para **CopyAir**: el motor SOTA de **Image-to-Image translation** orientado a VFX, Restauración y De-Aging.

### 📌 ¿Qué incluye este Notebook?
1. **Setup Acelerado por GPU & Sincronización Directa con Google Drive Desktop** (`/content/drive/Othercomputers/My Computer/Tesis/datasets`).
2. **Detección Automática de Datasets de Tesis**: `deaging`, `wireremoval`, `car_reflejo`, `reflections`.
3. **Catálogo SOTA Oficial**: NAFNet, Restormer (MDTA), ScopeUNet (Lossless PixelUnshuffle + ICNR), ConvNeXt-V2, MambaIR (SS2D), Residual U-Net.
4. **Configurador de Hiperparámetros y Pérdidas Híbridas** (LPIPS, Charbonnier, DINO, Laplacian).
5. **MLflow Offline/Local** (`sqlite:///mlflow.db`): Registra métricas, parámetros y checkpoints en `mlruns/` sin depender de servidores en vivo.
6. **Estudio de Inferencia HD/4K (Tiled)** para procesar imágenes, secuencias y videos completos sin Out-Of-Memory (OOM).

---
## 1. ⚙️ Hardware Check & Montaje de Google Drive

Verificamos la GPU asignada (T4, V100, A100 o L4) y montamos Google Drive. Tus datasets sincronizados desde la PC mediante **Google Drive Desktop** estarán disponibles de inmediato.

In [ ]:
# 1. Comprobar GPU
!nvidia-smi

import torch
print(f"\n🔥 PyTorch Version: {torch.__version__}")
print(f"🚀 CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🎮 GPU Name: {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"💾 VRAM Total: {vram_gb:.2f} GB")
else:
    print("⚠️ ADVERTENCIA: No se detectó GPU. Ve a Entorno de ejecución -> Cambiar tipo de entorno -> T4 GPU")

In [ ]:
# 2. Montar Google Drive
import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("✓ Google Drive montado exitosamente en /content/drive")
except Exception as e:
    print(f"Nota: {e}")

---
## 2. 📦 Clonar Repositorio e Instalar Dependencias

Usamos `uv` para una instalación ultrarrápida y determinista del código sincronizado de CopyAir y sus dependencias.

In [ ]:
import os
from pathlib import Path

# Clonar el repositorio si no existe en el directorio de trabajo
REPO_DIR = Path("/content/copyair")
if not REPO_DIR.exists():
    !git clone https://github.com/Serces19/copyair.git /content/copyair
    %cd /content/copyair
else:
    %cd /content/copyair
    !git pull

# Instalar uv y requerimientos
!pip install -q uv
!uv pip install --system -r requirements.txt

print("\n✓ Entorno y dependencias de CopyAir instaladas exitosamente!")

---
## 3. 📂 Explorador y Detector Automático de Datasets en Google Drive

Esta celda escanea automáticamente las rutas de **Google Drive Desktop** para localizar tus datasets (`deaging`, `wireremoval`, `car_reflejo`, `reflections`).

In [ ]:
import glob
from pathlib import Path

# Rutas candidatas habituales en Google Drive Desktop
POSSIBLE_ROOTS = [
    Path("/content/drive/Othercomputers/My Computer/Tesis/datasets"),
    Path("/content/drive/Othercomputers/Mi PC/Tesis/datasets"),
    Path("/content/drive/MyDrive/Tesis/datasets"),
    Path("/content/drive/MyDrive/datasets"),
    Path("data/01_raw")
]

DATASET_ROOT_FOUND = None
for root in POSSIBLE_ROOTS:
    if root.exists():
        DATASET_ROOT_FOUND = root
        break

if DATASET_ROOT_FOUND is None:
    # Búsqueda dinámica en Othercomputers
    other_comps = list(glob.glob("/content/drive/Othercomputers/*/**/datasets", recursive=True))
    if other_comps:
        DATASET_ROOT_FOUND = Path(other_comps[0])

print(f"📁 Directorio de Datasets detectado: {DATASET_ROOT_FOUND}\n")

if DATASET_ROOT_FOUND and DATASET_ROOT_FOUND.exists():
    datasets = [d for d in DATASET_ROOT_FOUND.iterdir() if d.is_dir()]
    print("📊 Datasets disponibles:")
    for d in datasets:
        in_p = d / "input"
        gt_p = d / "gt"
        ext_p = d / "extracted_frames"
        n_in = len(list(in_p.glob("*.png")) + list(in_p.glob("*.jpg"))) if in_p.exists() else 0
        n_gt = len(list(gt_p.glob("*.png")) + list(gt_p.glob("*.jpg"))) if gt_p.exists() else 0
        n_ext = len(list(ext_p.glob("*.png")) + list(ext_p.glob("*.jpg"))) if ext_p.exists() else 0
        print(f"  • [{d.name}]: {n_in} pares de entrenamiento (input/gt) | {n_ext} frames extraídos de video")
else:
    print("⚠️ No se encontró la carpeta automática. Podrás escribir la ruta manualmente abajo.")

---
## 4. 🧠 Guía de Modelos y Generador Interactivo de Configuración

Selecciona la arquitectura y el dataset para generar `configs/params.yaml`:

| Modelo | Variante recomendada | Fortalezas principales |
|---|---|---|
| **`nafnet`** | `base` / `small` | **Recomendado SOTA General**. Sin activaciones complejas (SimpleGate), LayerScale, ultrarrápido y preserva texturas finas. |
| **`restormer`** | `small` / `tiny` | **Recomendado SOTA Transformer**. Multi-Dconv Head Transposed Attention (MDTA) con complejidad $O(HW)$ + GDFN. |
| **`scope_unet`** | `base` / `small` | **Anti-Checkerboard**. Downsample sin pérdida con `PixelUnshuffle` + Upsample `ICNR PixelShuffle`. |
| **`convnext`** | `tiny` / `nano` | Gran campo receptivo (7x7 depthwise), regularización DropPath, decoder con GroupNorm + GRN. |
| **`mambair`** | `base` / `tiny` | State-Space Model 2D (SS2D) en 4 direcciones transversales con memoria lineal. |
| **`residual_unet`** | `base_channels: 64` | Baseline robusto y rápido con Pre-Act Mish, GroupNorm, Dilated Bottleneck y atajos de identidad puros. |

> **Nota sobre MLflow:** La configuración se genera automáticamente con `tracking_uri: "sqlite:///mlflow.db"`, registrando todos los runs en `./mlruns` de forma 100% nativa y sin necesidad de abrir puertos ni servidores.

In [ ]:
import yaml
from pathlib import Path

# =========================================================================
# 🎛️ 1. SELECCIÓN DE DATASET (GOOGLE DRIVE DESKTOP O LOCAL)
# =========================================================================
DATASET_NAME = "deaging"  # @param ["deaging", "wireremoval", "car_reflejo", "reflections", "custom"]
CUSTOM_DATASET_ROOT = "/content/drive/Othercomputers/My Computer/Tesis/datasets"  # @param {type:"string"}

# Resolver directorios
if DATASET_NAME != "custom":
    base_p = Path(CUSTOM_DATASET_ROOT) / DATASET_NAME
    if not base_p.exists() and DATASET_ROOT_FOUND:
        base_p = DATASET_ROOT_FOUND / DATASET_NAME
    INPUT_DIR = str(base_p / "input")
    GT_DIR = str(base_p / "gt")
    FRAMES_DIR = str(base_p / "extracted_frames")
else:
    INPUT_DIR = f"{CUSTOM_DATASET_ROOT}/input"
    GT_DIR = f"{CUSTOM_DATASET_ROOT}/gt"
    FRAMES_DIR = f"{CUSTOM_DATASET_ROOT}/extracted_frames"

OUTPUT_DIR = f"output_inference/{DATASET_NAME}"

# =========================================================================
# 🎛️ 2. SELECTOR DE ARQUITECTURA Y PRESETS SOTA
# =========================================================================
# Opciones de ARCHITECTURE: "nafnet", "restormer", "scope_unet", "convnext", "mambair", "residual_unet", "modern_unet", "smart_unet", "basic_unet"
SELECTED_ARCH = "nafnet"  # @param ["nafnet", "restormer", "scope_unet", "convnext", "mambair", "residual_unet", "modern_unet", "smart_unet", "basic_unet"]
MODEL_SIZE = "base"       # @param ["nano", "tiny", "small", "base", "large"]
EPOCHS = 1000             # @param {type:"integer"}
BATCH_SIZE = 4            # @param {type:"integer"}
LEARNING_RATE = 0.001     # @param {type:"number"}
IMG_SIZE = 512            # @param {type:"integer"}

# Construir diccionario de configuración base
config = {
    "data": {
        "input_dir": INPUT_DIR,
        "gt_dir": GT_DIR,
        "frames_dir": FRAMES_DIR if Path(FRAMES_DIR).exists() else INPUT_DIR,
        "output_dir": OUTPUT_DIR,
        "models_dir": "models"
    },
    "model": {
        "architecture": SELECTED_ARCH,
        "size": MODEL_SIZE,
        "in_channels": 3,
        "out_channels": 3,
        "dropout_p": 0.05
    },
    "training": {
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "weight_decay": 1e-4,
        "mixed_precision": True,
        "gradient_accumulation_steps": 1,
        "clear_cache_every_n_epochs": 10,
        "optimizer": {
            "type": "adamw",
            "beta1": 0.9,
            "beta2": 0.999
        },
        "scheduler": {
            "type": "cosine",
            "params": {
                "T_max": EPOCHS,
                "eta_min": 1e-6
            }
        },
        "val_split": 0,
        "val_interval": 50,
        "val_samples": 4,
        "viz_interval": 100,
        "save_interval": 250,
        "early_stopping_patience": 500
    },
    "loss": {
        "lambda_charbonnier": 0.2,
        "lambda_perceptual": 0.8,
        "lambda_dino": 0.2,
        "lambda_laplacian": 0.1,
        "lambda_ssim": 0.0
    },
    "augmentation": {
        "enabled": True,
        "img_size": IMG_SIZE,
        "horizontal_flip_p": 0.5,
        "vertical_flip_p": 0.0,
        "rotation_limit": 10,
        "gaussian_noise_p": 0.0,
        "gaussian_blur_p": 0.05,
        "color_jitter_p": 0.05
    },
    "device": "cuda",
    "num_workers": 2,
    "pin_memory": True,
    "logging": {
        "level": "INFO",
        "log_dir": "logs"
    },
    "mlflow": {
        "enabled": True,
        "experiment_name": f"copyair_{DATASET_NAME}_{SELECTED_ARCH}_{MODEL_SIZE}",
        "tracking_uri": "sqlite:///mlflow.db"
    },
    "inference": {
        "target_fps": 24,
        "frame_sample_rate": 1
    }
}

# Ajustes específicos por arquitectura
if SELECTED_ARCH == "nafnet":
    config["model"]["size"] = MODEL_SIZE if MODEL_SIZE in ["small", "base", "large"] else "base"
elif SELECTED_ARCH == "restormer":
    config["model"]["size"] = MODEL_SIZE if MODEL_SIZE in ["tiny", "small", "base"] else "small"
    config["training"]["learning_rate"] = 3e-4
elif SELECTED_ARCH in ["scope_unet", "gated_unet"]:
    config["model"]["size"] = MODEL_SIZE if MODEL_SIZE in ["small", "base"] else "base"
elif SELECTED_ARCH == "convnext":
    config["model"]["size"] = MODEL_SIZE if MODEL_SIZE in ["nano", "tiny", "small", "base"] else "tiny"
    config["model"]["drop_path_rate"] = 0.10
elif SELECTED_ARCH in ["mambair", "vmamba"]:
    config["model"]["size"] = MODEL_SIZE if MODEL_SIZE in ["tiny", "base", "large"] else "base"
    config["training"]["learning_rate"] = 5e-4
elif SELECTED_ARCH == "residual_unet":
    config["model"].update({
        "base_channels": 64,
        "norm_type": "group",
        "activation": "mish",
        "groups": 32,
        "residual": {"use_dilated_bottleneck": True}
    })
elif SELECTED_ARCH == "modern_unet":
    config["model"].update({
        "base_channels": 64,
        "norm_type": "group",
        "activation": "silu",
        "modern": {
            "attention_type": "self"
        }
    })
elif SELECTED_ARCH == "smart_unet":
    config["model"].update({
        "base_channels": 64,
        "smart": {
            "use_attention": True,
            "use_smart_filter": False
        }
    })

# Guardar en configs/params.yaml
Path("configs").mkdir(parents=True, exist_ok=True)
with open("configs/params.yaml", "w", encoding="utf-8") as f:
    yaml.dump(config, f, sort_keys=False, default_flow_style=False)

print(f"✓ Dataset seleccionado: {DATASET_NAME.upper()}")
print(f"  • Input dir:  {INPUT_DIR}")
print(f"  • GT dir:     {GT_DIR}")
print(f"  • Frames dir: {FRAMES_DIR}")
print(f"✓ Arquitectura: {SELECTED_ARCH.upper()} ({MODEL_SIZE}) -> configs/params.yaml")
print(f"✓ MLflow Tracking configurado offline: sqlite:///mlflow.db (mlruns)")

---
## 5. 🔍 Inspección Visual del Dataset Seleccionado

Visualizamos los pares de entrada (`Input`) y objetivo (`Ground Truth`) antes de entrenar para verificar calidad, alineación de frames y dimensiones.

In [ ]:
import glob
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from src.data.dataset import PairedImageDataset

in_dir = config["data"]["input_dir"]
gt_dir = config["data"]["gt_dir"]

try:
    ds = PairedImageDataset(input_dir=in_dir, gt_dir=gt_dir)
    print(f"📊 Total pares coincidentes cargados exitosamente: {len(ds)}\n")
    
    n_show = min(3, len(ds))
    fig, axes = plt.subplots(n_show, 2, figsize=(12, 4 * n_show))
    if n_show == 1:
        axes = [axes]
        
    for i in range(n_show):
        item = ds[i]
        # Convertir de tensor [-1, 1] (C, H, W) a uint8 [0, 255] (H, W, C)
        img_in = ((item["input"].permute(1, 2, 0).numpy() + 1.0) * 127.5).clip(0, 255).astype("uint8")
        img_gt = ((item["gt"].permute(1, 2, 0).numpy() + 1.0) * 127.5).clip(0, 255).astype("uint8")
        
        in_name = item.get('input_name', item.get('filename', f'frame_{i}'))
        gt_name = item.get('gt_name', item.get('filename', f'frame_{i}'))
        
        axes[i][0].imshow(img_in)
        axes[i][0].set_title(f"Input: {in_name} ({img_in.shape[1]}x{img_in.shape[0]})", fontsize=11)
        axes[i][0].axis("off")
        
        axes[i][1].imshow(img_gt)
        axes[i][1].set_title(f"Ground Truth: {gt_name} ({img_gt.shape[1]}x{img_gt.shape[0]})", fontsize=11)
        axes[i][1].axis("off")
        
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"❌ Error al cargar dataset: {e}")

---
## 6. 🚀 Ejecución del Entrenamiento

Lanzamos el script `scripts/train.py` en GPU. Los mejores pesos se guardarán automáticamente en `models/<run_id>/best_model_<arch>.pth` y los logs se escribirán en `mlruns` y `sqlite:///mlflow.db`.

In [ ]:
# Ejecutar entrenamiento con la configuración generada
!python scripts/train.py --config configs/params.yaml --device cuda

---
## 7. 📈 Análisis de Curvas de Pérdida y Métricas (MLflow Local)

Leemos las métricas registradas de forma 100% local desde `mlruns/` para visualizar la convergencia del entrenamiento (Loss, PSNR, SSIM, LPIPS).

In [ ]:
import mlflow
from mlflow.tracking import MlflowClient
import matplotlib.pyplot as plt
import pandas as pd

# Conectar con el MLflow SQLite local
mlflow.set_tracking_uri("sqlite:///mlflow.db")
client = MlflowClient("sqlite:///mlflow.db")

experiments = client.search_experiments()
print(f"Experimentos encontrados: {[e.name for e in experiments]}")

if experiments:
    exp = experiments[0]
    runs = client.search_runs(experiment_ids=[exp.experiment_id], order_by=["start_time DESC"])
    
    if runs:
        latest_run = runs[0]
        run_id = latest_run.info.run_id
        print(f"📊 Visualizando métricas del Run: {latest_run.info.run_name} ({run_id})")
        metrics_to_plot = [
            ("train/loss", "LOSS TOTAL", "royalblue"),
            ("train/lpips", "LPIPS PERCEPTUAL LOSS", "crimson"),
            ("train/dino", "DINOv2 SEMANTIC LOSS", "purple"),
            ("train/charbonnier", "CHARBONNIER PIXEL LOSS", "darkorange"),
            ("train/laplacian", "LAPLACIAN TEXTURE LOSS", "forestgreen"),
            ("val/psnr", "VALIDATION PSNR (dB)", "teal")
        ]
        
        fig, axes = plt.subplots(2, 3, figsize=(18, 9))
        axes = axes.flatten()
        
        for idx, (m_name, m_title, m_color) in enumerate(metrics_to_plot):
            try:
                history = client.get_metric_history(run_id, m_name)
                if not history and m_name == "train/lpips":
                    history = client.get_metric_history(run_id, "train/perceptual")
                if history:
                    steps = [m.step for m in history]
                    vals = [m.value for m in history]
                    axes[idx].plot(steps, vals, label=m_name, color=m_color, lw=2.2)
                    axes[idx].set_title(m_title, fontsize=12, fontweight="bold")
                    axes[idx].set_xlabel("Época")
                    axes[idx].grid(True, alpha=0.3)
                else:
                    axes[idx].text(0.5, 0.5, f"Sin datos: {m_name}", ha="center", fontsize=11)
                    axes[idx].set_title(m_title, fontsize=12, color="gray")
            except Exception as e:
                axes[idx].text(0.5, 0.5, f"No disponible: {m_name}", ha="center", fontsize=11)
                axes[idx].set_title(m_title, fontsize=12, color="gray")
                
        plt.tight_layout()
        plt.show()
    else:
        print("No hay runs completados en este experimento aún.")

---
## 8. 🔮 Inferencia HD/4K (Secuencia de Video Completa)

Aplica el modelo entrenado sobre la secuencia completa extraída del video (`extracted_frames`).

### 💎 Modo Tiled (Recomendado para 2K / 4K):
Al activar `--tiled`, el frame se divide en parches con solapamiento suave, permitiendo procesar cualquier resolución sin saturar la VRAM de la GPU.

In [ ]:
import glob
import os
from pathlib import Path

# Buscar el mejor modelo primero (best_model), ignorando checkpoints interrumpidos
best_candidates = sorted(glob.glob("models/**/best_model_*.pth", recursive=True) + glob.glob("models/best_model_*.pth"), key=os.path.getmtime, reverse=True)
if best_candidates:
    LATEST_MODEL = best_candidates[0]
    print(f"🎯 Mejor modelo detectado (Best Loss): {LATEST_MODEL}")
else:
    all_candidates = sorted(glob.glob("models/**/*.pth", recursive=True), key=os.path.getmtime, reverse=True)
    LATEST_MODEL = all_candidates[0] if all_candidates else f"models/best_model_{SELECTED_ARCH}.pth"
    print(f"🎯 Modelo detectado: {LATEST_MODEL}")

# =========================================================================
# 🎛️ CONFIGURACIÓN DE INFERENCIA
# =========================================================================
MODEL_PATH = LATEST_MODEL            # @param {type:"string"}
INPUT_MEDIA = config["data"]["frames_dir"] # @param {type:"string"}
OUTPUT_MEDIA = config["data"]["output_dir"] # @param {type:"string"}
USE_TILED = True                     # @param {type:"boolean"}
TILE_SIZE = 512                      # @param {type:"integer"}
OVERLAP = 64                         # @param {type:"integer"}

tiled_flag = f"--tiled --tile-size {TILE_SIZE} --overlap {OVERLAP}" if USE_TILED else ""

!python scripts/predict.py \
    --config configs/params.yaml \
    --model {MODEL_PATH} \
    --video "{INPUT_MEDIA}" \
    --output "{OUTPUT_MEDIA}" \
    --native-resolution \
    --device cuda \
    {tiled_flag}

---
## 9. 🖼️ Visualizador Interactivo Antes / Después (Side-by-Side)

Comparación visual inmediata del frame de entrada vs el resultado generado por CopyAir.

In [ ]:
import glob
import cv2
import os
import matplotlib.pyplot as plt
from pathlib import Path

# Buscar salidas generadas de forma flexible
candidate_dirs = [
    Path(config["data"]["output_dir"]),
    Path(f"output_inference/{DATASET_NAME}"),
    Path(f"output_inference/{DATASET_NAME}_{SELECTED_ARCH}")
] + [d for d in Path("output_inference").glob("*") if d.is_dir()]

out_frames = []
selected_output_dir = None

for c_dir in candidate_dirs:
    if c_dir.exists() and c_dir.is_dir():
        found = sorted(list(c_dir.glob("*.png")) + list(c_dir.glob("*.jpg")) + list(c_dir.glob("*.jpeg")))
        if found:
            out_frames = found
            selected_output_dir = c_dir
            break

video_files = list(Path("output_inference").glob("**/*.mp4")) + list(Path("output_inference").glob("**/*.mov"))

if out_frames:
    in_dir = Path(config["data"]["frames_dir"])
    in_frames = sorted(list(in_dir.glob("*.png")) + list(in_dir.glob("*.jpg")) + list(in_dir.glob("*.jpeg")))
    
    idx = 0
    img_out = cv2.cvtColor(cv2.imread(str(out_frames[idx])), cv2.COLOR_BGR2RGB)
    
    fig, ax = plt.subplots(1, 2, figsize=(16, 8))
    if in_frames:
        img_in = cv2.cvtColor(cv2.imread(str(in_frames[min(idx, len(in_frames)-1)])), cv2.COLOR_BGR2RGB)
        ax[0].imshow(img_in)
        ax[0].set_title(f"ORIGINAL / INPUT ({Path(in_frames[0]).name})", fontsize=14)
        ax[0].axis("off")
    else:
        ax[0].axis("off")
        
    ax[1].imshow(img_out)
    ax[1].set_title(f"COPYAIR RESULT ({out_frames[idx].name}) en {selected_output_dir.name}", fontsize=14)
    ax[1].axis("off")
    
    plt.tight_layout()
    plt.show()
    print(f"✓ Total frames generados: {len(out_frames)} en {selected_output_dir}")
elif video_files:
    print(f"✓ Video generado exitosamente en: {video_files[0]}")
else:
    print("No se encontraron salidas generadas en output_inference/")

---
## 10. 💾 Sincronización Automática con Google Drive

Guarda los pesos entrenados (`models/`), la base de datos de experimentos (`mlruns` / `mlflow.db`) y los resultados generados directamente en tu carpeta de **Google Drive**, permitiendo que aparezcan al instante en tu ordenador local vía **Google Drive Desktop**.

In [ ]:
import shutil
from pathlib import Path

# Ruta destino en Google Drive (sincronizada con tu PC)
DRIVE_BACKUP_DIR = Path("/content/drive/Othercomputers/My Computer/Tesis/datasets") / DATASET_NAME / "training_runs"
if not DRIVE_BACKUP_DIR.parent.exists():
    DRIVE_BACKUP_DIR = Path("/content/drive/MyDrive/CopyAir_Backups") / DATASET_NAME

DRIVE_BACKUP_DIR.mkdir(parents=True, exist_ok=True)
print(f"📁 Sincronizando hacia: {DRIVE_BACKUP_DIR} ...\n")

# 1. Copiar modelos
if Path("models").exists():
    shutil.copytree("models", DRIVE_BACKUP_DIR / "models", dirs_exist_ok=True)
    print("✓ Modelos (.pth) sincronizados con Google Drive.")

# 2. Copiar base de datos y experimentos MLflow
if Path("mlruns").exists():
    shutil.copytree("mlruns", DRIVE_BACKUP_DIR / "mlruns", dirs_exist_ok=True)
if Path("mlflow.db").exists():
    shutil.copy2("mlflow.db", DRIVE_BACKUP_DIR / "mlflow.db")
print("✓ Experimentos MLflow sincronizados con Google Drive.")

# 3. Copiar outputs de inferencia (toda la carpeta output_inference)
if Path("output_inference").exists():
    shutil.copytree("output_inference", DRIVE_BACKUP_DIR / "output_inference", dirs_exist_ok=True)
    print("✓ Resultados de inferencia sincronizados con Google Drive.")

print("\n🎉 ¡Sincronización completada! Tus archivos ya están disponibles en tu PC vía Google Drive Desktop.")